# Generative Models

Notebook for **Chapter 13** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 Autoencoders

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
# 3-D data that really lives near a 2-D plane
Z = rng.standard_normal((500, 2)) * np.array([3.0, 1.0])
B = np.array([[1., 0., 1.], [0., 1., -1.]]) / np.sqrt(2)
X = Z @ B + 0.1 * rng.standard_normal((500, 3))
X = X - X.mean(0)
def train_ae(k, steps=3000, lr=0.01):
    r = np.random.default_rng(1)
    We = r.standard_normal((3, k)) * 0.3; Wd = r.standard_normal((k, 3)) * 0.3
    for t in range(steps):
        H = X @ We; Xh = H @ Wd; E = Xh - X
        gWd = H.T @ E / len(X); gWe = X.T @ (E @ Wd.T) / len(X)
        Wd -= lr * gWd; We -= lr * gWe
    return ((X @ We @ Wd - X) ** 2).mean()
print("total variance per feature", (X**2).mean().round(4))
for k in (1, 2, 3): print("bottleneck", k, "reconstruction MSE", round(train_ae(k), 4))
U, S, Vt = np.linalg.svd(X, full_matrices=False)
for k in (1, 2): print("PCA k", k, "MSE", round(((X - X @ Vt[:k].T @ Vt[:k])**2).mean(), 4))

total variance per feature 3.2139
bottleneck 1 reconstruction MSE 0.2329
bottleneck 2 reconstruction MSE 0.0036
bottleneck 3 reconstruction MSE 0.0037
PCA k 1 MSE 0.2329
PCA k 2 MSE 0.0034


## 13.2 Variational Autoencoders (VAE)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
mu, sigma = 1.0, 0.5
kl = 0.5 * (sigma**2 + mu**2 - 1 - np.log(sigma**2)); print("KL by formula", round(kl, 4))
eps = rng.standard_normal(1_000_000); z = mu + sigma * eps        # the reparameterisation
logq = -0.5*np.log(2*np.pi*sigma**2) - (z-mu)**2/(2*sigma**2)
logp = -0.5*np.log(2*np.pi) - z**2/2
print("KL by sampling", round((logq - logp).mean(), 4))
print("gradient d/dmu E[z^2] via reparam:", round((2*z).mean(), 3), " exact:", 2*mu)
print("gradient wrt sigma via reparam:", round((2*z*eps).mean(), 3), " exact:", 2*sigma)
for m, s in [(0, 1), (0, 0.1), (3, 1)]: print("mu", m, "sigma", s, "KL", round(0.5*(s**2 + m**2 - 1 - np.log(s**2)), 4))
mu2 = np.array([0.5, -1.0]); lv = np.array([-0.2, 0.4])
print("2-D KL", round((0.5*(np.exp(lv) + mu2**2 - 1 - lv)).sum(), 4))

KL by formula 0.8181
KL by sampling 0.8181
gradient d/dmu E[z^2] via reparam: 2.001  exact: 2.0
gradient wrt sigma via reparam: 1.003  exact: 1.0
mu 0 sigma 1 KL 0.0
mu 0 sigma 0.1 KL 1.8076
mu 3 sigma 1 KL 4.5
2-D KL 0.6803


In [ ]:
# not run here
mu, logvar = encoder(x)                       # two outputs
z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)    # reparameterisation
loss = F.mse_loss(decoder(z), x, reduction="sum") + (-0.5 * torch.sum(1 + logvar - mu**2 - logvar.exp()))

## 13.3 Generative Adversarial Networks (GANs)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
sig = lambda t: 1/(1+np.exp(-t))
a, b = 1.0, 0.0                      # generator g(z) = a*z + b
w = np.zeros(3)                      # discriminator D(x) = sigmoid(w0*x + w1*x^2 + w2)
feat = lambda x: np.stack([x, x**2, np.ones_like(x)], 1)
lrD, lrG, N = 0.02, 0.02, 128
hist = []
for step in range(6001):
    real = rng.normal(4.0, 1.25, N); z = rng.standard_normal(N); fake = a*z + b
    # discriminator step: maximise log D(real) + log(1-D(fake))
    Fr, Ff = feat(real), feat(fake)
    gD = Fr.T @ (1 - sig(Fr@w)) / N - Ff.T @ sig(Ff@w) / N
    w += lrD * gD
    # generator step (non-saturating): maximise log D(fake)
    z = rng.standard_normal(N); fake = a*z + b; Ff = feat(fake)
    dlogit = (1 - sig(Ff@w))                      # d log D / d logit
    dx = dlogit * (w[0] + 2*w[1]*fake)            # chain to x
    a += lrG * (dx * z).mean(); b += lrG * dx.mean()
    if step % 1000 == 0:
        hist.append((step, round(float(b),3), round(float(abs(a)),3)))
print(hist)
fake = a*rng.standard_normal(100000)+b; print("final generated mean", fake.mean().round(3), "std", fake.std().round(3), " target 4.0 and 1.25")

[(0, 0.0, 1.002), (1000, 4.016, 0.541), (2000, 4.172, 0.081), (3000, 4.186, 0.035), (4000, 4.189, 0.043), (5000, 4.196, 0.141), (6000, 3.968, 1.107)]
final generated mean 3.97 std 1.104  target 4.0 and 1.25


## 13.4 Diffusion Models

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
T = 1000
beta = np.linspace(1e-4, 0.02, T); alpha = 1 - beta; abar = np.cumprod(alpha)
for t in (1, 100, 250, 500, 750, 1000):
    i = t - 1; print("t", t, "abar", round(abar[i], 5), "signal scale", round(np.sqrt(abar[i]), 4), "noise scale", round(np.sqrt(1-abar[i]), 4))
# closed form vs step by step, for x0 = 2.0 at t = 300
rng = np.random.default_rng(0)
t = 300
M = 200000
xt_iter = np.full(M, 2.0)
for i in range(t): xt_iter = np.sqrt(alpha[i]) * xt_iter + np.sqrt(beta[i]) * rng.standard_normal(M)
xt_closed = np.sqrt(abar[t-1]) * 2.0 + np.sqrt(1-abar[t-1]) * rng.standard_normal(M)
print("iterative  mean/std", xt_iter.mean().round(3), xt_iter.std().round(3))
print("closed form mean/std", xt_closed.mean().round(3), xt_closed.std().round(3))
# forward noising of a 1-D "image" row
sig = np.sin(np.linspace(0, 64*np.pi, 4096))
for t in (1, 100, 300, 600, 1000):
    e = rng.standard_normal(4096); xt = np.sqrt(abar[t-1])*sig + np.sqrt(1-abar[t-1])*e
    ab=abar[t-1]; print("t", t, "corr(x_t, x_0) =", round(np.corrcoef(xt, sig)[0, 1], 3), " theory", round(np.sqrt(ab*0.5/(ab*0.5+1-ab)), 3))

t 1 abar 0.9999 signal scale 0.9999 noise scale 0.01
t 100 abar 0.89702 signal scale 0.9471 noise scale 0.3209
t 250 abar 0.52409 signal scale 0.7239 noise scale 0.6899
t 500 abar 0.07859 signal scale 0.2803 noise scale 0.9599
t 750 abar 0.00335 signal scale 0.0579 noise scale 0.9983
t 1000 abar 4e-05 signal scale 0.0064 noise scale 1.0
iterative  mean/std 1.262 0.778
closed form mean/std 1.261 0.775
t 1 corr(x_t, x_0) = 1.0  theory 1.0
t 100 corr(x_t, x_0) = 0.901  theory 0.902
t 300 corr(x_t, x_0) = 0.486  theory 0.497
t 600 corr(x_t, x_0) = 0.096  theory 0.114
t 1000 corr(x_t, x_0) = 0.02  theory 0.004


In [ ]:
# continues the script above (it uses T, beta, alpha, abar, rng)
mu0, s0 = 2.0, 0.5
def eps_hat(x, t):
    ab = abar[t]; var = ab*s0**2 + 1 - ab
    return np.sqrt(1-ab) * (x - np.sqrt(ab)*mu0) / var
M = 100000; x = rng.standard_normal(M)
for t in range(T-1, -1, -1):
    ab = abar[t]; mean = (x - beta[t]/np.sqrt(1-ab) * eps_hat(x, t)) / np.sqrt(alpha[t])
    x = mean + (np.sqrt(beta[t]) * rng.standard_normal(M) if t > 0 else 0)
print("generated mean", x.mean().round(3), "std", x.std().round(3), "target 2.0 and 0.5")

generated mean 1.998 std 0.501 target 2.0 and 0.5


In [ ]:
# not run here
t = torch.randint(0, T, (batch,))                      # random step for each image
eps = torch.randn_like(x0)                             # the noise we add
xt = abar[t].sqrt().view(-1,1,1,1) * x0 + (1 - abar[t]).sqrt().view(-1,1,1,1) * eps
loss = F.mse_loss(model(xt, t), eps)                   # predict the noise

In [ ]:
e_u, e_c = np.array([0.10, -0.30]), np.array([0.40, -0.10])
for w in (0, 1, 3, 7.5): print("w", w, "guided eps", e_u + w*(e_c - e_u))
print("pixel space", 512*512*3, "latent", 64*64*4, "ratio", 512*512*3/(64*64*4))

w 0 guided eps [ 0.1 -0.3]
w 1 guided eps [ 0.4 -0.1]
w 3 guided eps [1.  0.3]
w 7.5 guided eps [2.35 1.2 ]
pixel space 786432 latent 16384 ratio 48.0


## 13.5 Diffusion in More Depth: Derivation, Schedules, DDIM, Latent Space, and Stabler GANs

Section 13.4 gave you the forward formula, the reverse loop and classifier-free guidance. This section answers the questions an expert is asked: where the formula comes from, why the noise schedule matters, how sampling gets 20 times faster, why Stable-Diffusion-style models work in a compressed space, and how GAN training was made less fragile.

In [ ]:
import numpy as np
T = 1000
beta = np.linspace(1e-4, 0.02, T); abar_lin = np.cumprod(1-beta)
s = 0.008
tt = np.arange(T+1)/T
f = np.cos((tt+s)/(1+s)*np.pi/2)**2
abar_cos = (f/f[0])[1:]
print(" t   abar_lin  abar_cos   SNR_lin   SNR_cos")
for t in (1, 100, 250, 500, 750, 900, 1000):
    a, c = abar_lin[t-1], abar_cos[t-1]
    print(f"{t:4d}  {a:8.4f}  {c:8.4f}  {a/(1-a):9.4f} {c/(1-c):9.4f}")

 t   abar_lin  abar_cos   SNR_lin   SNR_cos
   1    0.9999    1.0000  9999.0000 24221.3272
 100    0.8970    0.9721     8.7104   34.8330
 250    0.5241    0.8470     1.1012    5.5365
 500    0.0786    0.4938     0.0853    0.9757
 750    0.0034    0.1443     0.0034    0.1686
 900    0.0003    0.0241     0.0003    0.0247
1000    0.0000    0.0000     0.0000    0.0000


In [ ]:
import numpy as np
T = 1000
beta = np.linspace(1e-4, 0.02, T); abar = np.cumprod(1-beta)
x0, eps = 2.0, 0.5
t, t_prev = 250, 200
xt = np.sqrt(abar[t-1])*x0 + np.sqrt(1-abar[t-1])*eps
x0_hat = (xt - np.sqrt(1-abar[t-1])*eps)/np.sqrt(abar[t-1])
x_prev = np.sqrt(abar[t_prev-1])*x0_hat + np.sqrt(1-abar[t_prev-1])*eps
print("abar_250 =", round(abar[t-1],4), " abar_200 =", round(abar[t_prev-1],4))
print("x_250 =", round(xt,4), " x0_hat =", round(x0_hat,4), " x_200 (DDIM) =", round(x_prev,4))
direct = np.sqrt(abar[t_prev-1])*x0 + np.sqrt(1-abar[t_prev-1])*eps
print("direct forward jump to t=200 with same eps =", round(direct,4))
print("DDPM 1000 steps vs DDIM 50 steps: speedup", 1000/50)

abar_250 = 0.5241  abar_200 = 0.659
x_250 = 1.7928  x0_hat = 2.0  x_200 (DDIM) = 1.9156
direct forward jump to t=200 with same eps = 1.9156
DDPM 1000 steps vs DDIM 50 steps: speedup 20.0


In [ ]:
h, w, c = 512, 512, 3
lh, lw, lc = 64, 64, 4
pix, lat = h*w*c, lh*lw*lc
print("pixel numbers:", f"{pix:,}", " latent numbers:", f"{lat:,}", " ratio:", pix/lat)
print("spatial positions: pixel", f"{h*w:,}", " latent", f"{lh*lw:,}", " ratio:", (h*w)/(lh*lw))
print("self-attention score pairs if run at each resolution:", f"{(h*w)**2:,}", "vs", f"{(lh*lw)**2:,}", " ratio:", ((h*w)**2)//((lh*lw)**2))

pixel numbers: 786,432  latent numbers: 16,384  ratio: 48.0
spatial positions: pixel 262,144  latent 4,096  ratio: 64.0
self-attention score pairs if run at each resolution: 68,719,476,736 vs 16,777,216  ratio: 4096


In [ ]:
import numpy as np
real = np.array([2.0, 1.0, 1.5]); fake = np.array([-0.5, 0.5, 0.0])
print("critic gap (Wasserstein estimate):", real.mean()-fake.mean())
print("critic loss:", -(real.mean()-fake.mean()), " generator loss:", 0.0 - fake.mean())
lam = 10
for g in [0.5, 1.0, 2.0]:
    print(f"gradient norm {g}: penalty {lam*(g-1)**2:.2f}")
rng = np.random.default_rng(0)
ang = np.arange(8)*2*np.pi/8
centers = np.c_[np.cos(ang), np.sin(ang)]*2
def sample(modes, n=800):
    k = rng.choice(modes, n); return centers[k] + 0.05*rng.normal(size=(n,2))
def covered(x, thresh=0.01):
    d = np.linalg.norm(x[:,None,:]-centers[None], axis=2).argmin(1)
    share = np.bincount(d, minlength=8)/len(x)
    return int((share >= thresh).sum()), share.round(3)
print("healthy generator :", covered(sample(range(8))))
print("collapsed (2 modes):", covered(sample([0, 5])))

critic gap (Wasserstein estimate): 1.5
critic loss: -1.5  generator loss: 0.0
gradient norm 0.5: penalty 2.50
gradient norm 1.0: penalty 0.00
gradient norm 2.0: penalty 10.00
healthy generator : (8, array([0.126, 0.105, 0.115, 0.12 , 0.131, 0.132, 0.13 , 0.14 ]))
collapsed (2 modes): (2, array([0.528, 0.   , 0.   , 0.   , 0.   , 0.472, 0.   , 0.   ]))


## Chapter recap

## Practice set

**1.** (easy) An autoencoder maps 100 inputs to 10 codes and back with one dense layer each way (biases included). How many parameters? **Answer:** encoder `100·10 + 10 = 1,010`; decoder `10·100 + 100 = 1,100`; total `2,110`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch13.html